# 125.785 Research and Analytics in Economics and Finance
## Session 5: Panel data I: pooled OLS, fixed and random effects

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nquynh/125785-research-analytics/blob/main/notebooks/S05_student.ipynb)

**Reading:** Brooks (2019) Ch. 11; Python Guide Ch. 21  
**Data:** `vn_bank_panel.csv`

**By the end of this session you can:**
- Set up a panel index in pandas
- Estimate pooled OLS, fixed effects (entity and time) and random effects with linearmodels
- Choose between FE and RE (Hausman test)
- Use lagged regressors and clustered standard errors

*All datasets are simulated for teaching. They mimic the structure of Vietnamese listed firms, banks and markets, but are not real company data.*

### 0. Setup

In [ ]:
# Run this cell first in every session
!pip -q install linearmodels        # statsmodels, pandas, numpy, matplotlib are pre-installed on Colab
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

# Where the course data live (GitHub). If you uploaded the CSV files to Colab, use DATA = ""
DATA = os.environ.get("DATA_DIR", "https://raw.githubusercontent.com/nquynh/125785-research-analytics/main/data/")

### 1. Worked example

Research question: **Does rapid credit growth lead to higher non-performing loans two years later?** The data cover 27 simulated Vietnamese banks, 2010–2024. Model: npl_it = β1·credit_growth_i,t−2 + β2·car_it + β3·size_it + α_i + λ_t + u_it.

In [ ]:
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS, compare
bank = pd.read_csv(DATA + "vn_bank_panel.csv").sort_values(["bank_id", "year"])
bank["cg_l2"] = bank.groupby("bank_id")["credit_growth"].shift(2)     # 2-year lag
bank = bank.dropna(subset=["cg_l2"]).set_index(["bank_id", "year"])  # entity-time index
print(bank.shape)
bank.head()

In [ ]:
exog = sm.add_constant(bank[["cg_l2", "car", "size"]])
pooled = PooledOLS(bank["npl"], exog).fit(cov_type="clustered", cluster_entity=True)
fe = PanelOLS(bank["npl"], exog, entity_effects=True).fit(cov_type="clustered", cluster_entity=True)
fe2 = PanelOLS(bank["npl"], exog, entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
re = RandomEffects(bank["npl"], exog).fit()
print(compare({"Pooled": pooled, "FE": fe, "FE + year": fe2, "RE": re}, stars=True))

### 2. Exercises

#### Exercise 5.1 – Interpretation
In the two-way FE model (bank and year effects), interpret the coefficient on `cg_l2` in units. If a bank's credit growth is 10 pp higher, how much higher is its NPL ratio two years later?

In [ ]:
# YOUR CODE HERE


**Your interpretation:** _write 2–3 sentences here_

#### Exercise 5.2 – Hausman test
Write a function for the Hausman test comparing FE (entity effects only) and RE on the slope coefficients. Which model do you choose?

In [ ]:
# YOUR CODE HERE


**Your interpretation:** _write 2–3 sentences here_

#### Exercise 5.3 – Why the lag?
Re-estimate the two-way FE model with contemporaneous credit growth (`credit_growth`) instead of `cg_l2`. Compare the coefficients and explain why the lagged specification is more sensible.

In [ ]:
# YOUR CODE HERE


**Your interpretation:** _write 2–3 sentences here_

---
**Before you leave:** File → Save a copy in Drive, then File → Download → .ipynb. Your notebook *is* your log: keep it with your A1/A2 working files.